# 1. Veri Ön İşleme ve Temizleme (Data Preprocessing & Cleaning)
* Eksik verilerin medyan ve mod ile doldurulması
* Kategorik değişkenlerin One-Hot Encoding ile dönüştürülmesi

1. Adımı: Veriyi Yükleme ve Boyutları Görme

In [ ]:
import pandas as pd
import numpy as np

# Veri setlerini içeri alalım
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

# Matris boyutlarını (Satır x Sütun) görelim
print(f"Eğitim (Train) verisi boyutu: {train.shape}")
print(f"Test verisi boyutu: {test.shape}")

# Verinin ilk 3 satırına bakarak yapıyı inceleyelim
display(train.head(3))

In [ ]:
# 1. Eğitim setinde eksik veri (NaN) barındıran sütunları ve sayılarını görelim
eksik_veriler = train.isnull().sum()
print("Eksik veri içeren sütunlar ve miktarları:")
print(eksik_veriler[eksik_veriler > 0])

print("-" * 40)

# 2. Sayısal değişkenlerin temel istatistikleri (Min, Max, Ortalama vb.)
# Sadece virgülden sonra 2 basamak göstererek tabloyu okunabilir yapalım
display(train.describe().round(2).T)

Adım 2: Görselleştirme ve Korelasyon (Hangi değişkenler gerçekten önemli?)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sadece sayısal sütunları seçelim
sayisal_kolonlar = train.select_dtypes(include=['float64', 'int64']).columns

# Korelasyon matrisini hesapla
korelasyon_matrisi = train[sayisal_kolonlar].corr()

# Görselleştirme (Isı Haritası)
plt.figure(figsize=(16, 12))
# Sadece hedef değişkene göre sıralanmış korelasyonları görelim ki işimiz kolaylaşsın
hedef_korelasyonu = korelasyon_matrisi[['bilissel_performans_skoru']].sort_values(by='bilissel_performans_skoru', ascending=False)

sns.heatmap(hedef_korelasyonu, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title("Değişkenlerin Bilişsel Performans Skoru ile Korelasyonu")
plt.show()

Eksik Verilerin Doldurulması (Imputation)


In [ ]:
# 1. Sayısal ve kategorik sütunları tespit edelim
# (id ve hedef değişkeni işleme dahil etmiyoruz)
sayisal_kolonlar = train.select_dtypes(include=['float64', 'int64']).columns
sayisal_kolonlar = sayisal_kolonlar.drop(['id', 'bilissel_performans_skoru'])

kategorik_kolonlar = train.select_dtypes(include=['object']).columns

# 2. Sayısal boşlukları EĞİTİM setinin medyanı ile dolduralım
for col in sayisal_kolonlar:
    medyan_degeri = train[col].median()
    train[col] = train[col].fillna(medyan_degeri)
    test[col] = test[col].fillna(medyan_degeri)

# 3. Kategorik boşlukları EĞİTİM setinin modu (en çok tekrar eden değeri) ile dolduralım
for col in kategorik_kolonlar:
    mod_degeri = train[col].mode()[0]
    train[col] = train[col].fillna(mod_degeri)
    test[col] = test[col].fillna(mod_degeri)

# 4. İşlemin başarıya ulaştığını kontrol edelim
print("Eksik veri doldurma işlemi tamamlandı.")
print(f"Train setinde kalan eksik veri toplamı: {train.isnull().sum().sum()}")
print(f"Test setinde kalan eksik veri toplamı: {test.isnull().sum().sum()}")
train.head(10)

Kategorik Değişkenlerin Dönüştürülmesi (One-Hot Encoding)


In [ ]:
# 1. Kategorik kolonları tespit edelim (işlemi bağımsızlaştırmak için tekrar tanımlıyoruz)
kategorik_kolonlar = train.select_dtypes(include=['object']).columns

# 2. One-Hot Encoding işlemi (n-1 kuralı ile)
train_encoded = pd.get_dummies(train, columns=kategorik_kolonlar, drop_first=True)
test_encoded = pd.get_dummies(test, columns=kategorik_kolonlar, drop_first=True)

# 3. Train ve Test setlerindeki sütun yapılarını hizalayalım
# (Test setinde olmayan kategoriler için oluşan sütunları 0 ile doldurur)
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# 4. Hizalama sırasında test setine yanlışlıkla hedef değişken eklendiyse onu kaldıralım
if 'bilissel_performans_skoru' in test_encoded.columns:
    test_encoded = test_encoded.drop(columns=['bilissel_performans_skoru'])

# İşlem sonucu oluşan matris boyutlarını kontrol edelim
print(f"Encoding öncesi Train sütun sayısı: {train.shape[1]}")
print(f"Encoding sonrası Train boyutu: {train_encoded.shape}")
print("-" * 40)
print(f"Encoding öncesi Test sütun sayısı: {test.shape[1]}")
print(f"Encoding sonrası Test boyutu: {test_encoded.shape}")

# 2. Özellik Mühendisliği (Feature Engineering)
* Biyolojik ve mantıksal yeni özelliklerin türetilmesi
* Aykırı değerlerin (Outliers) işaretlenmesi


In [ ]:
def özellik_muhendisligi(df):
    df_yeni = df.copy()
    
    # 1. Aykırı Değer İşaretleme (Outlier Flagging)
    # Kafein tüketimi 300mg üzeri olanları ve BMI 30 üzeri (Obez) olanları işaretleyelim
    df_yeni['is_extreme_caffeine'] = (df_yeni['uyku_oncesi_kafein_mg'] > 300).astype(int)
    df_yeni['is_obese'] = (df_yeni['vucut_kitle_indeksi'] > 30).astype(int)
    
    # 2. Dinlendirici Uyku Oranı
    # REM ve Derin uykunun toplamının, genel uykuya katkısı
    df_yeni['dinlendirici_uyku_orani'] = df_yeni['rem_yuzdesi'] + df_yeni['derin_uyku_yuzdesi']
    
    # 3. Zihinsel Yıpranma Endeksi (Bilişsel Yük)
    # Stres skoru ile günlük çalışma saatinin çarpan etkisi
    df_yeni['bilissel_yuk_endeksi'] = df_yeni['stres_skoru'] * df_yeni['gunluk_calisma_saati']
    
    # 4. Uyku Hijyeni İhlali
    # Uyumadan önceki ekran süresi ile kafeinin yıkıcı kombinasyonu
    df_yeni['uyku_hijyeni_ihlali'] = df_yeni['uyku_oncesi_kafein_mg'] * df_yeni['uyku_oncesi_ekran_suresi_dk']
    
    # 5. Gece Huzursuzluğu
    # Uykuya dalma süresi ile uyanma sayısının çarpımı (Kalitesiz gece şiddeti)
    df_yeni['gece_huzursuzlugu'] = df_yeni['uykuya_dalma_suresi_dk'] * df_yeni['gecelik_uyanma_sayisi']

    return df_yeni

# Fonksiyonu hem eğitim hem test setimize uygulayalım
train_fe = özellik_muhendisligi(train_encoded)
test_fe = özellik_muhendisligi(test_encoded)

print(f"Feature Engineering öncesi Train sütun sayısı: {train_encoded.shape[1]}")
print(f"Feature Engineering sonrası Train sütun sayısı: {train_fe.shape[1]}")

# 3. Modelleme İçin Veri Hazırlığı (X ve y Ayrımı)
* Hedef değişkenin ve ID sütunlarının özellik matrisinden ayrıştırılması

In [ ]:
# Train setinden özellikleri (X) ve hedef değişkeni (y) ayıralım
X = train_fe.drop(columns=['id', 'bilissel_performans_skoru'])
y = train_fe['bilissel_performans_skoru']

# Test setinden sadece özellikleri alalım (Kaggle'a tahmin üreteceğimiz matris)
# Submission dosyasında kullanmak üzere test_id'leri güvenli bir yere kopyalayalım
test_id = test_fe['id']
X_test = test_fe.drop(columns=['id'])

# Boyutları son kez kontrol edelim (X ve X_test sütun sayıları tamamen aynı olmalı!)
print(f"Eğitim Matrisi (X) Boyutu: {X.shape}")
print(f"Hedef Değişken (y) Boyutu: {y.shape}")
print(f"Test Matrisi (X_test) Boyutu: {X_test.shape}")

# 4. Çapraz Doğrulama ve Temel Model (Cross-Validation & Baseline Model)
* K-Fold ile 5 katlı doğrulama
* LightGBM Regressor ile ilk RMSE skorunun hesaplanması

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

# 5 katlı çapraz doğrulama yapısını kuruyoruz
# shuffle=True ile veriyi bölmeden önce iyice karıştırıyoruz
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Her bir katmanın (fold) RMSE skorunu tutacağımız liste
rmse_skorlari = []

# Katmanları saymak için basit bir sayaç
fold = 1

# K-Fold döngüsü
for train_index, val_index in kf.split(X):
    # Veriyi eğitim ve doğrulama (validation) olarak ayır
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]
    
    # LightGBM Modelini tanımla (Şimdilik hiperparametre ayarı yapmıyoruz)
    # n_jobs=-1 bilgisayarın tüm işlemci çekirdeklerini kullanmasını sağlar
    model = LGBMRegressor(random_state=42, n_jobs=-1)
    
    # Modeli eğit
    model.fit(X_train_fold, y_train_fold)
    
    # Doğrulama seti üzerinde tahmin yap
    tahminler = model.predict(X_val_fold)
    
    # RMSE (Kök-Ortalama-Kare-Hata) hesapla
    hata = np.sqrt(mean_squared_error(y_val_fold, tahminler))
    rmse_skorlari.append(hata)
    
    print(f"Fold {fold} RMSE Skoru: {hata:.4f}")
    fold += 1

print("-" * 30)
# En önemli metrik: 5 katmanın ortalama hatası
ortalama_rmse = np.mean(rmse_skorlari)
print(f"Ortalama (Baseline) RMSE Skoru: {ortalama_rmse:.4f}")